In [1]:
import numpy as np
import ot_utils
import comparison_utils
from tqdm import tqdm
import math
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score


n_classes = 8
labels = np.repeat(np.arange(8), 1000)

w2_block_files = []

for i in range(n_classes):
    for j in range(i, n_classes):
        w2_block_files.append(f"../../mini_w2/block_{i}_{j}.npy")

w2_matrix = ot_utils.combine_distance_blocks(w2_block_files)
ps_matrix = np.load("../../mini_ps/8k_ps_dist_matrix.npy")

In [2]:
first_class = 4
second_class = 5

idx_list_1 = list(range(first_class * 1000, (first_class + 1) * 1000))
idx_list_2 = list(range(second_class * 1000, (second_class + 1) * 1000))

idx_list = idx_list_1 + idx_list_2

In [3]:
w2_1_vs_2 = comparison_utils.slice_from_dist_matrix(w2_matrix, idx_list=idx_list)
ps_1_vs_2 = comparison_utils.slice_from_dist_matrix(ps_matrix, idx_list=idx_list)

labels = np.repeat([0, 1], [1000, 1000])

In [4]:
permutation = np.random.permutation(2000)

w2_1_vs_2 = w2_1_vs_2[permutation, :][:, permutation]
ps_1_vs_2 = ps_1_vs_2[permutation, :][:, permutation]

labels = labels[permutation]

In [5]:
train_length = 2000 // 5 * 4
test_length = 2000 // 5

w2_train_matrix = w2_1_vs_2[:train_length, :train_length]
ps_train_matrix = ps_1_vs_2[:train_length, :train_length]
print(w2_train_matrix.shape)
print(ps_train_matrix.shape)

w2_test_matrix = w2_1_vs_2[train_length:, :train_length]
ps_test_matrix = ps_1_vs_2[train_length:, :train_length]
print(w2_test_matrix.shape)
print(ps_test_matrix.shape)


train_labels = labels[:train_length]
test_labels = labels[train_length:]

neighbor_list = list(range(1, train_length // 2, 2))

w2_best_k = comparison_utils.kNNGridSearchCV(w2_train_matrix, train_labels, neighbor_list, nFolds=5)
ps_best_k = comparison_utils.kNNGridSearchCV(ps_train_matrix, train_labels, neighbor_list, nFolds=5)

w2_best_k = w2_best_k["n_neighbors"]
ps_best_k = ps_best_k["n_neighbors"]
print(w2_best_k)
print(ps_best_k)

w2_metric_dict, w2_avg_auc, w2_std_auc = comparison_utils.kNNPerformanceEvaluation(
    w2_train_matrix, w2_test_matrix, train_labels, test_labels, w2_best_k
)
ps_metric_dict, ps_avg_auc, ps_std_auc = comparison_utils.kNNPerformanceEvaluation(
    ps_train_matrix, ps_test_matrix, train_labels, test_labels, ps_best_k
)

(1600, 1600)
(1600, 1600)
(400, 1600)
(400, 1600)
193
31


In [6]:
print(f"AUC of W2: {w2_avg_auc:.3f} +- {w2_std_auc:.3f}")
print(f"AUC of ps: {ps_avg_auc:.3f} +- {ps_std_auc:.3f}")

AUC of W2: 0.892 +- 0.031
AUC of ps: 0.813 +- 0.055


In [7]:
w2_all_vs_all   = np.copy(w2_matrix)
ps_all_vs_all  = np.copy(ps_matrix)

labels = np.repeat([0, 1, 2, 3, 4, 5, 6, 7], [1000, 1000, 1000, 1000, 1000, 1000, 1000, 10000])

In [8]:
np.random.seed(42)
permutation = np.random.permutation(8000)

w2_all_vs_all = w2_all_vs_all[permutation, :][:, permutation]
ps_all_vs_all = ps_all_vs_all[permutation, :][:, permutation]

labels = labels[permutation]

In [9]:
train_length = 8000 // 5 * 4
test_length = 8000 // 5

w2_train_matrix = w2_all_vs_all[:train_length, :train_length]
ps_train_matrix = ps_all_vs_all[:train_length, :train_length]
print(w2_train_matrix.shape)
print(ps_train_matrix.shape)

w2_test_matrix = w2_all_vs_all[train_length:, :train_length]
ps_test_matrix = ps_all_vs_all[train_length:, :train_length]
print(w2_test_matrix.shape)
print(ps_test_matrix.shape)


train_labels = labels[:train_length]
test_labels = labels[train_length:]

neighbor_list = list(range(3, train_length//32, 4))

w2_best_k = comparison_utils.kNNGridSearchCV(w2_train_matrix, train_labels, neighbor_list, nFolds=5)
ps_best_k = comparison_utils.kNNGridSearchCV(ps_train_matrix, train_labels, neighbor_list, nFolds=5)

w2_best_k = w2_best_k["n_neighbors"]
ps_best_k = ps_best_k["n_neighbors"]
print(w2_best_k)
print(ps_best_k)

w2_avg_auc, w2_std_auc = comparison_utils.kNNPerformanceEvaluationAcc(
    w2_train_matrix, w2_test_matrix, train_labels, test_labels, w2_best_k
)
ps_avg_auc, ps_std_auc = comparison_utils.kNNPerformanceEvaluationAcc(
    ps_train_matrix, ps_test_matrix, train_labels, test_labels, ps_best_k
)

(6400, 6400)
(6400, 6400)
(1600, 6400)
(1600, 6400)
11
187


In [10]:
print(f"ACC of W2: {w2_avg_auc:.3f} +- {w2_std_auc:.3f}")
print(f"ACC of ps: {ps_avg_auc:.3f} +- {ps_std_auc:.3f}")

ACC of W2: 0.366 +- 0.017
ACC of ps: 0.395 +- 0.029
